In [18]:
from pathlib import Path

path_project_root = Path.cwd().parent
path_dataset = path_project_root.joinpath("data/proofwriter-dataset-V2020.12.3/OWA/depth-1")

In [19]:
import json
import random
import re

# Load the filtered subset (a JSON list of theories).
path_subset = path_project_root.joinpath(f"data/proofwriter-subset/meta-train.json")
with open(path_subset) as f:
    subset = json.load(f)

# Pick N_THEORIES theories at random. The same SEED always gives the same picks.
random.seed(42)
sample = random.sample(subset, 100)

path_sample = path_project_root.joinpath(f"data/proofwriter-subset/sample-train-seed42.json")
with open(path_sample, "w") as f:
    json.dump(sample, f, indent=2)

# Build one minimal data point per sampled theory:
# only the fact(s) and the rule of one proof, plus its two questions.
data_points = []

for theory in sample:
    # Both questions share the same proofs, so take them from the first one.
    first_question = list(theory["questions"].values())[0]
    proofs = first_question["proofs"][2:-2].split(" OR ")

    # Choose one proof at random. This runs after random.seed(42) above,
    # so the same proof is chosen every time the cell is run from scratch.
    proof = random.choice(proofs)

    # The names of the facts and the rule the proof uses,
    # e.g. ((triple8 triple6) -> rule3) gives triple8, triple6 and rule3.
    fact_names = re.findall(r"triple\d+", proof)
    rule_name = re.findall(r"rule\d+", proof)[0]

    # Facts in the order they appear in the original theory, then the rule.
    facts = {name: fact["text"] for name, fact in theory["triples"].items() if name in fact_names}
    rule = {rule_name: theory["rules"][rule_name]["text"]}

    data_points.append({
        # For tracing back: the theory id, the split it came from, and the chosen proof.
        "id": theory["id"],
        "split": "train",
        "proof": proof,
        # The premises: names point to the original triples and rules.
        "triples": facts,
        "rules": rule,
        # The new theory text: facts first, then the rule.
        "theory": " ".join(list(facts.values()) + list(rule.values())),
        # The True and False questions, unchanged, with their original names (e.g. Q3, Q4).
        "questions": theory["questions"],
    })

path_data_points = path_project_root.joinpath(f"data/proofwriter-subset/sample-train-seed42-no-clutter-single-proof.json")
with open(path_data_points, "w") as f:
    json.dump(data_points, f, indent=2)

In [20]:
import json

# Load the 100 data points (one minimal theory each).
path_data_points = path_project_root.joinpath("data/proofwriter-subset/sample-train-seed42-no-clutter-single-proof.json")
with open(path_data_points) as f:
    data_points = json.load(f)

# One document per data point in each list.
# Entry i of every list belongs to data point i.
premise_docs = []    # facts and rule together
true_docs = []       # C: the True question
false_docs = []      # !C: the False question

for data_point in data_points:
    # The theory field is already the facts followed by the rule.
    premise_docs.append(data_point["theory"])

    # Each data point has exactly one True and one False question.
    for question in data_point["questions"].values():
        if question["answer"] is True:
            true_docs.append(question["question"])
        else:
            false_docs.append(question["question"])

# Every text, for fit to build the vocabulary from.
fit_docs = premise_docs + true_docs + false_docs

In [21]:
from sklearn.feature_extraction.text import CountVectorizer

# Binary: 1 if a word is present, 0 if not.
binary_vectorizer = CountVectorizer(
    lowercase=True,                 # "Dave" and "dave" are the same word
    token_pattern=r"(?u)\b\w+\b",   # keep 1-letter words too (the default drops them)
    binary=True,
)

# Build the word list from every text.
binary_vectorizer.fit(fit_docs)

# Turn each list of documents into bags: one row per data point.
binary_premise_bags = binary_vectorizer.transform(premise_docs)
binary_true_bags = binary_vectorizer.transform(true_docs)
binary_false_bags = binary_vectorizer.transform(false_docs)

In [32]:
import random

n_data_points = len(true_docs)


# ---- Q: a conclusion from a different, randomly chosen data point ----

# The seed makes the same Q get picked every time the cell is run.
random.seed(42)

q_docs = []
for i in range(n_data_points):
    # Choose one of the other data points (never data point i itself).
    other_data_points = [j for j in range(n_data_points) if j != i]
    other = random.choice(other_data_points)

    # Q is that data point's True or False conclusion, chosen at random.
    q_docs.append(random.choice([true_docs[other], false_docs[other]]))

binary_q_bags = binary_vectorizer.transform(q_docs)


# ---- Coverage: share of a statement's words that appear in the premises ----

# Both bags are binary: 1 if a word is present, 0 if not.
# Multiplying them word by word gives 1 only for words present in both,
# so the sum of that is the number of shared words.
# The sum of the statement bag is the number of words in the statement.
def coverage(premise_bag, statement_bag):
    shared_words = statement_bag.multiply(premise_bag).sum()
    statement_words = statement_bag.sum()
    return shared_words / statement_words


coverage_true = []
coverage_false = []
coverage_q = []

for i in range(n_data_points):
    premise_bag = binary_premise_bags[i]
    coverage_true.append(coverage(premise_bag, binary_true_bags[i]))
    coverage_false.append(coverage(premise_bag, binary_false_bags[i]))
    coverage_q.append(coverage(premise_bag, binary_q_bags[i]))

# Print the id of every data point where Q's coverage is higher than C's or higher than !C's,
# and count how many there are.
q_higher_count = 0
for i in range(n_data_points):
    if coverage_q[i] > coverage_true[i] or coverage_q[i] > coverage_false[i]:
        print(f"{data_points[i]['id']}: {premise_docs[i]} {true_docs[i]} {false_docs[i]} \n                    C: {round(coverage_true[i], 3)}, !C: {round(coverage_false[i], 3)}, Q: {round(coverage_q[i], 3)} | Q: {q_docs[i]}")
        q_higher_count += 1

print("Number of Q > C or !C:", q_higher_count)

RelNeg-OWA-D1-2309: The cow is rough. If something is rough then it sees the cow. The cow sees the cow. The cow does not see the cow. 
                    C: 1.0, !C: 0.4, Q: 0.5 | Q: Fiona is not rough.
Number of Q > C or !C: 1


In [23]:
import pandas as pd

# The file goes next to the subset files.
path_results = path_project_root.joinpath("data/proofwriter-subset")

# The theory id of every data point, in the same order as the results.
ids = [data_point["id"] for data_point in data_points]

# Coverage: one row per data point.
coverage_table = pd.DataFrame({
    "id": ids,
    "C": coverage_true,
    "!C": coverage_false,
    "Q": coverage_q,
})
coverage_table = coverage_table.round(5)
coverage_table.to_csv(path_results.joinpath("bow-results-train-seed42-coverage.csv"), index_label="index")

In [24]:
import pandas as pd

# YES means every word of the statement is in the premises (coverage 1.0).
# Mean coverage is the average share of the statement's words found in the premises.
coverage_rows = []
for statement, values in [("C", coverage_true), ("!C", coverage_false), ("Q", coverage_q)]:
    yes_count = 0
    for value in values:
        if value == 1.0:
            yes_count += 1

    coverage_rows.append({
        "statement": statement,
        "YES": yes_count,
        "NO": len(values) - yes_count,
        "mean coverage": sum(values) / len(values),
    })

coverage_summary = pd.DataFrame(coverage_rows).round(5)
print("Coverage, out of", n_data_points, "data points")
print(coverage_summary.to_string(index=False))

Coverage, out of 100 data points
statement  YES  NO  mean coverage
        C   90  10        0.97433
       !C   12  88        0.74198
        Q    0 100        0.28033
